# Engine lab: make time move

This lab is for people who can read small Python classes and want to understand SimYuj's event engine by running it. The engine is the clock and scheduler: it owns time, event ids, ordering, batches, cancellation, summaries, statistics, and named random streams. Components own the meaning of actions and payloads.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from simyuj.engine import Component, Event, Timeline, event_ordering_key
from simyuj.tracing import LogLevel, MemorySink, SimulationLogger

## 1. One target, one event

A target only needs `handle_event(event, timeline)`. Start with a recorder that does nothing clever: it writes down what the timeline delivers.

In [ ]:
@dataclass(slots=True)
class Recorder(Component):
    name: str
    rows: list[tuple[int, int | None, str, object]] = field(default_factory=list)

    def handle_event(self, event, timeline) -> None:
        self.rows.append(
            (timeline.current_time, event.event_id, event.action, event.payload_ref)
        )

In [ ]:
timeline = Timeline(master_seed=11)
recorder = Recorder('sensor')

wake = Event(
    time=5,
    target_ref=recorder,
    action='wake',
    payload_ref={'note': 'first scheduled event'},
)

print('before scheduling, event_id =', wake.event_id)
scheduled = timeline.schedule(wake)
print('after scheduling, event_id =', scheduled.event_id)
print('recorder before execution:', recorder.rows)

In [ ]:
summary = timeline.run_one_step()

print('summary:', summary)
print('current time:', timeline.current_time)
print('recorder after one step:', recorder.rows)
print('stats:', timeline.stats)

## 2. Components talk by scheduling

The source below does not call the recorder. It schedules an arrival event and lets the timeline decide when that arrival is delivered.

In [ ]:
@dataclass(slots=True)
class Source(Component):
    name: str
    receiver: Recorder
    sent: list[str] = field(default_factory=list)

    def handle_event(self, event, timeline) -> None:
        message = event.payload_ref['message']
        delay = event.payload_ref['delay']
        arrival_time = timeline.current_time + delay

        self.sent.append(f'{message} leaves at t={timeline.current_time}')
        timeline.schedule(
            Event(
                time=arrival_time,
                target_ref=self.receiver,
                action='arrive',
                payload_ref={'message': message, 'from': self.name},
                source=self,
            )
        )

In [ ]:
receiver = Recorder('receiver')
source = Source('source', receiver)
timeline = Timeline(master_seed=11)

timeline.schedule(
    Event(
        time=3,
        target_ref=source,
        action='send',
        payload_ref={'message': 'pulse A', 'delay': 4},
    )
)

first_batch = timeline.run_one_step()
print('first batch:', first_batch)
print('source log:', source.sent)
print('receiver is still waiting:', receiver.rows)

In [ ]:
rest = timeline.run_until_empty()

print('later batches:', rest)
print('receiver after the timeline finishes:')
for row in receiver.rows:
    print(row)

## 3. Predict same-time order

For events at the same time, the ordering key is `(time, priority, event_id)`. Lower priority runs first; if priority ties, the event scheduled earlier wins.

In [ ]:
ordered = Recorder('ordered')
timeline = Timeline(master_seed=5)

slow = timeline.schedule(Event(time=10, priority=5, target_ref=ordered, action='slow', payload_ref=None))
first = timeline.schedule(Event(time=10, priority=0, target_ref=ordered, action='first', payload_ref=None))
second = timeline.schedule(Event(time=10, priority=0, target_ref=ordered, action='second', payload_ref=None))
urgent = timeline.schedule(Event(time=10, priority=-2, target_ref=ordered, action='urgent', payload_ref=None))

for event in [slow, first, second, urgent]:
    print(event.action, 'has key', event_ordering_key(event))

In [ ]:
summary = timeline.run_one_step()

print('batch summary:', summary)
print('actions in delivery order:', [row[2] for row in ordered.rows])

## 4. A batch closes before handlers run

An event handler may schedule more work at the current tick. That new event is valid, but it waits for the next batch.

In [ ]:
@dataclass(slots=True)
class SameTickSpawner(Component):
    log: list[tuple[int, int | None, str]] = field(default_factory=list)

    def handle_event(self, event, timeline) -> None:
        self.log.append((timeline.current_time, event.event_id, event.action))
        if event.action == 'start':
            timeline.schedule(
                Event(
                    time=timeline.current_time,
                    target_ref=self,
                    action='spawned-at-same-time',
                    payload_ref=None,
                )
            )

In [ ]:
spawner = SameTickSpawner()
timeline = Timeline()
timeline.schedule(Event(time=8, target_ref=spawner, action='start', payload_ref=None))

first = timeline.run_one_step()
print('after first batch:', first)
print('log:', spawner.log)

second = timeline.run_one_step()
print('after second batch:', second)
print('log:', spawner.log)
print('current time stayed at:', timeline.current_time)

## 5. Stop at a time boundary

`run_until(t)` executes whole batches with timestamps up to and including `t`. Later batches remain queued.

In [ ]:
receiver = Recorder('run-until')
timeline = Timeline()

for tick in [2, 4, 6]:
    timeline.schedule(
        Event(
            time=tick,
            target_ref=receiver,
            action=f'arrive-{tick}',
            payload_ref={'tick': tick},
        )
    )

timeline.run_until(4)
print('current time after run_until(4):', timeline.current_time)
print('delivered so far:', [row[2] for row in receiver.rows])

finish = timeline.run_until_empty()
print('remaining summaries:', finish)
print('all delivered:', [row[2] for row in receiver.rows])

## 6. Cancel and reschedule

Cancellation is lazy inside the queue, but the visible result is simple: a cancelled event is skipped. Rescheduling cancels the old event and creates a fresh event id.

In [ ]:
recorder = Recorder('cancellation')
timeline = Timeline()

old = timeline.schedule(Event(time=5, target_ref=recorder, action='old-plan', payload_ref=None))
replacement = timeline.reschedule(old, new_time=2, new_priority=-1)

to_drop = timeline.schedule(Event(time=3, target_ref=recorder, action='drop-me', payload_ref=None))
timeline.cancel(to_drop)

timeline.run_until_empty()

print('old event:', {'id': old.event_id, 'cancelled': old.cancelled})
print('replacement:', {'id': replacement.event_id, 'cancelled': replacement.cancelled})
print('dropped event:', {'id': to_drop.event_id, 'cancelled': to_drop.cancelled})
print('delivered actions:', [row[2] for row in recorder.rows])
print('stats still count scheduled work:', timeline.stats)

## 7. Named random streams

Create the streams before execution begins. The path, not the order you ask for paths, decides the stream.

In [ ]:
def first_draws(paths):
    timeline = Timeline(master_seed=123)
    streams = {}
    for path in paths:
        streams[path] = timeline.rng(*path)
    return {'/'.join(path): round(stream.random(), 6) for path, stream in streams.items()}

paths_a = [('alice', 'basis'), ('bob', 'basis')]
paths_b = [('bob', 'basis'), ('alice', 'basis')]

print('alice then bob:', first_draws(paths_a))
print('bob then alice:', first_draws(paths_b))

In [ ]:
@dataclass(slots=True)
class Dice(Component):
    name: str
    rng: object | None = None
    rolls: list[tuple[int, int]] = field(default_factory=list)

    def handle_event(self, event, timeline) -> None:
        roll = self.rng.randint(1, 6)
        self.rolls.append((timeline.current_time, roll))


dice = Dice('detector')
timeline = Timeline(master_seed=77)
dice.rng = timeline.rng('dice', dice.name)

for tick in [1, 2, 3]:
    timeline.schedule(Event(time=tick, target_ref=dice, action='roll', payload_ref=None))

timeline.run_until_empty()
print('rolls:', dice.rolls)

try:
    timeline.rng('late', 'stream')
except RuntimeError as exc:
    print('new stream after execution:', exc)

print('existing stream still draws:', dice.rng.randint(1, 6))

## 8. Let the timeline narrate itself

Tracing is observational. It records what happened without changing event order.

In [ ]:
sink = MemorySink()
logger = SimulationLogger(level=LogLevel.TRACE, sinks=[sink])
timeline = Timeline(master_seed=3, logger=logger)
recorder = Recorder('logged')

timeline.schedule(Event(time=1, target_ref=recorder, action='note', payload_ref='hello'))
timeline.run_until_empty()

for record in sink.records[:6]:
    print(record.category, '| t =', record.sim_time, '| action =', record.action, '|', record.message)

## 9. Mini lab: a lossy link

Change `drop_on`, `base_delay`, or `pulse_count`, then rerun the next two cells. Watch how the same engine rules still explain every delivery.

In [ ]:
@dataclass(slots=True)
class LossyLink(Component):
    receiver: Recorder
    rng: object | None = None
    notes: list[str] = field(default_factory=list)

    def handle_event(self, event, timeline) -> None:
        pulse = event.payload_ref['pulse']
        base_delay = event.payload_ref['base_delay']
        drop_on = event.payload_ref['drop_on']
        roll = self.rng.randint(1, 6)

        if roll == drop_on:
            self.notes.append(f't={timeline.current_time}: dropped {pulse} on roll {roll}')
            return

        jitter = roll % 2
        arrival_time = timeline.current_time + base_delay + jitter
        self.notes.append(f't={timeline.current_time}: {pulse} roll={roll}, arrives at t={arrival_time}')
        timeline.schedule(
            Event(
                time=arrival_time,
                target_ref=self.receiver,
                action='receive-pulse',
                payload_ref={'pulse': pulse, 'roll': roll},
                source=self,
            )
        )

In [ ]:
pulse_count = 6
base_delay = 3
drop_on = 1

receiver = Recorder('terminal')
link = LossyLink(receiver)
timeline = Timeline(master_seed=2024)
link.rng = timeline.rng('link', 'loss')

for pulse_id in range(pulse_count):
    timeline.schedule(
        Event(
            time=pulse_id * 2,
            target_ref=link,
            action='transmit',
            payload_ref={
                'pulse': f'pulse-{pulse_id}',
                'base_delay': base_delay,
                'drop_on': drop_on,
            },
        )
    )

summaries = timeline.run_until_empty()

print('link decisions:', len(link.notes))
for note in link.notes[:4]:
    print(' ', note)
print('delivered rows:', receiver.rows)
print('batch times:', [summary.batch_time for summary in summaries])
print('final stats:', timeline.stats)


## Keep this model in your head

If you can answer these questions from the printed output, you understand the engine layer: who assigned the event id, why did that event run first, which batch did it belong to, did a component schedule future work, and where did randomness come from?